# Private RLHF: Best-of-N Runtime

This notebook benchmarks the generation and reward-scoring time of Best-of-`N` inference. It reads the private reward-model artifact selected from `master_results.csv` and writes all benchmark outputs to a timestamped folder under `ROOT/bon_latency`.

## Install dependencies

Install the packages used by the benchmark.

In [ ]:
# ============================================================
# Cell 1. Install dependencies
# ============================================================
!pip -q install "transformers>=4.40.0" datasets accelerate safetensors

## Paths and benchmark configuration

Change only the `ROOT` line when the training outputs are stored in a different Drive folder.

In [ ]:
# ============================================================
# Cell 2. Imports, Drive mount, paths, config, and safety checks
# ============================================================
import os, re, json, time, csv, hashlib, gc, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------------
# Existing project paths.
# These are read-only inputs except for Hugging Face cache writes.
# ------------------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/Private_Finetuning")
MASTER_CSV = ROOT / "master_results.csv"
ARTIFACTS_ROOT = ROOT / "artifacts"
CACHE_ROOT = ROOT / "cache"
HF_CACHE = CACHE_ROOT / "hf"

# ------------------------------------------------------------------
# New output path.
# All benchmark outputs are written only inside this timestamped folder.
# ------------------------------------------------------------------
OUT_ROOT = ROOT / "bon_latency"
RUN_TS = time.strftime("%Y%m%d_%H%M%S")

# Model and artifact choice
MODEL_ID = "google/gemma-2b-it"
EPS_FOR_LATENCY = 1.0
RM_SEED_FOR_ARTIFACT = 11

# Prompt sampling
N_TOTAL = 40000
TEST_FRAC = 0.2
PROMPT_SOURCE_SEED = 11
BENCHMARK_SEED = 2026  # controls which held-out prompts are sampled for timing
NUM_WARMUP_PROMPTS = 3
NUM_TIMED_PROMPTS = 50
MAX_PROMPT_TOKENS_FOR_LATENCY = 768

# Decoding and BoN setup used in the mixed-temperature illustration.
N_GRID = [2, 4, 8, 16, 32]
T_LOW = 0.2
T_HIGH = 0.8
TOP_P = 0.9
MAX_NEW_TOKENS = 160
DIRECT_TEMPERATURE = T_HIGH
DIRECT_TOP_P = TOP_P

# RM scoring / generation batching
RM_MAX_LEN = 256
RM_SCORE_BATCH_SIZE = 32
GEN_RETURN_SEQ_BATCH = 16  # A100 should handle 16; lower to 8 only if generation OOMs.

# Output naming
RUN_ID = (
    f"bon_latency_eps{EPS_FOR_LATENCY}_rmseed{RM_SEED_FOR_ARTIFACT}_"
    f"nprompt{NUM_TIMED_PROMPTS}_{RUN_TS}"
)
RUN_DIR = OUT_ROOT / RUN_ID

PER_PROMPT_CSV = RUN_DIR / "bon_latency_per_prompt.csv"
SUMMARY_CSV = RUN_DIR / "bon_latency_summary.csv"
CONFIG_JSON = RUN_DIR / "bon_latency_config.json"
LATEX_TABLE_TXT = RUN_DIR / "bon_latency_latex_table.txt"

# ------------------------------------------------------------------
# Safety checks before any writes.
# ------------------------------------------------------------------
if not ROOT.exists():
    raise FileNotFoundError(f"ROOT does not exist: {ROOT}")

if not MASTER_CSV.exists():
    raise FileNotFoundError(
        f"Cannot find MASTER_CSV: {MASTER_CSV}\n"
        "Please confirm ROOT points to the same Drive folder used by your training notebook."
    )

if not ARTIFACTS_ROOT.exists():
    raise FileNotFoundError(f"Cannot find artifacts folder: {ARTIFACTS_ROOT}")

# Create only cache/output folders. No deletion or overwrite of existing result files.
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Fail if this exact timestamped run folder already exists.
# This avoids accidentally overwriting a previous benchmark run.
if RUN_DIR.exists():
    raise FileExistsError(f"RUN_DIR already exists; re-run Cell 2 to create a new timestamp: {RUN_DIR}")
RUN_DIR.mkdir(parents=False, exist_ok=False)

# Hugging Face cache location.
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "GPU is required for meaningful latency. Please use a Colab GPU runtime."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("DEVICE       =", DEVICE)
print("ROOT         =", ROOT)
print("READ master  =", MASTER_CSV)
print("READ artifacts from:", ARTIFACTS_ROOT)
print("HF cache     =", HF_CACHE)
print("WRITE only to:", RUN_DIR)
print("SUMMARY_CSV  =", SUMMARY_CSV)
print("\nSafety note: this notebook does not delete files, does not edit master_results.csv, and does not write to artifacts/.")

## Benchmark utilities

In [ ]:
# ============================================================
# Cell 3. Helper functions
# ============================================================
PROMPT_PAT = re.compile(r"\n\nAssistant:\s*")

def extract_prompt_prefix(text: str) -> str:
    ms = list(PROMPT_PAT.finditer(text))
    if not ms:
        return ""
    return text[: ms[-1].end()]

def sha1_12(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:12]

def cleanup_gpu_only(*objs):
    """Free Python/GPU memory only. This does not delete files from Drive."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def timed_call(fn, *args, **kwargs):
    sync_cuda()
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    sync_cuda()
    t1 = time.perf_counter()
    return out, float(t1 - t0)

def latest_ok_rm_artifact(master_df: pd.DataFrame, eps: float, seed: int):
    required = ["method", "epsilon_target", "seed", "train_status", "eval_status", "run_id", "artifact_dir"]
    missing = [c for c in required if c not in master_df.columns]
    if missing:
        raise ValueError(f"MASTER_CSV is missing required columns: {missing}")

    sub = master_df[
        (master_df["method"].astype(str) == "dp_rm_postproc") &
        (pd.to_numeric(master_df["epsilon_target"], errors="coerce") == float(eps)) &
        (pd.to_numeric(master_df["seed"], errors="coerce") == int(seed)) &
        (master_df["train_status"].astype(str) == "ok") &
        (master_df["eval_status"].astype(str) == "ok")
    ].copy()
    if sub.empty:
        return None

    if "timestamp" in sub.columns:
        sub["ts"] = pd.to_datetime(sub["timestamp"], errors="coerce")
    else:
        sub["ts"] = pd.NaT
    sub = sub.sort_values(["ts", "run_id"], ascending=True)
    row = sub.iloc[-1]
    return str(row["run_id"]), Path(str(row["artifact_dir"]))

def load_rm_head_only(artifact_dir: Path, tokenizer, dtype=torch.float16):
    meta_path = artifact_dir / "rm_head_meta.json"
    head_path = artifact_dir / "rm_head.pt"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing RM metadata: {meta_path}")
    if not head_path.exists():
        raise FileNotFoundError(f"Missing RM head weights: {head_path}")

    with open(meta_path, "r") as f:
        meta = json.load(f)
    head_attr = meta["head_attr"]
    model_id = meta["model_id"]

    rm = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        torch_dtype=dtype,
        cache_dir=str(HF_CACHE),
    ).to(DEVICE)
    rm.config.pad_token_id = tokenizer.pad_token_id

    for p in rm.parameters():
        p.requires_grad_(False)

    head = getattr(rm, head_attr)
    state_dict = torch.load(head_path, map_location="cpu")
    head.load_state_dict(state_dict)
    rm.eval()
    return rm

@torch.inference_mode()
def generate_candidates(model, tokenizer, prompt: str, n: int, temperature: float, top_p: float, max_new: int, seed: int):
    """Generate n completions. Returns completion texts and generated-token counts."""
    if n <= 0:
        return [], []

    all_comps, all_counts = [], []
    remaining = int(n)
    chunk_id = 0

    while remaining > 0:
        cur_n = min(remaining, GEN_RETURN_SEQ_BATCH)
        cur_seed = int(seed + 97 * chunk_id)
        torch.manual_seed(cur_seed)
        np.random.seed(cur_seed)

        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        input_len = inputs["input_ids"].shape[1]

        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=float(temperature),
            top_p=float(top_p),
            num_return_sequences=cur_n,
            max_new_tokens=int(max_new),
            pad_token_id=tokenizer.eos_token_id,
        )

        for i in range(out.shape[0]):
            comp_ids = out[i, input_len:]
            all_comps.append(tokenizer.decode(comp_ids, skip_special_tokens=True).lstrip())
            all_counts.append(int((comp_ids != tokenizer.pad_token_id).sum().item()))

        remaining -= cur_n
        chunk_id += 1

    return all_comps, all_counts

@torch.inference_mode()
def score_candidates(rm, tokenizer, prompt: str, completions: list):
    """Score completions with the private reward model in batches."""
    if len(completions) == 0:
        return np.array([], dtype=np.float32)

    scores = []
    for start in range(0, len(completions), RM_SCORE_BATCH_SIZE):
        chunk = completions[start:start + RM_SCORE_BATCH_SIZE]
        texts = [prompt + c for c in chunk]
        enc = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=RM_MAX_LEN,
        ).to(DEVICE)
        s = rm(**enc).logits.squeeze(-1).detach().float().cpu().numpy()
        scores.append(s)

    return np.concatenate(scores, axis=0).astype(np.float32)

def run_direct_once(pi0, tokenizer, prompt: str, seed: int):
    comps, tok_counts = generate_candidates(
        pi0, tokenizer, prompt,
        n=1,
        temperature=DIRECT_TEMPERATURE,
        top_p=DIRECT_TOP_P,
        max_new=MAX_NEW_TOKENS,
        seed=seed,
    )
    return comps[0], tok_counts[0]

def run_bon_generation_once(pi0, tokenizer, prompt: str, N: int, seed_base: int):
    N_low = N // 2
    N_high = N - N_low

    comps_low, tok_low = generate_candidates(
        pi0, tokenizer, prompt,
        n=N_low,
        temperature=T_LOW,
        top_p=TOP_P,
        max_new=MAX_NEW_TOKENS,
        seed=seed_base + 1,
    )
    comps_high, tok_high = generate_candidates(
        pi0, tokenizer, prompt,
        n=N_high,
        temperature=T_HIGH,
        top_p=TOP_P,
        max_new=MAX_NEW_TOKENS,
        seed=seed_base + 2,
    )
    return comps_low + comps_high, tok_low + tok_high, N_low, N_high

def se(x):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan
    return float(x.std(ddof=1) / math.sqrt(len(x)))

def make_latex_table(summary_df: pd.DataFrame) -> str:
    df = summary_df.copy()
    order = {"direct_pi0": 0, "bon_rerank": 1}
    df["order"] = df["method"].map(order).fillna(99)
    df = df.sort_values(["order", "N_total"])

    lines = []
    lines.append(r"\begin{tabular}{lccccc}")
    lines.append(r"\toprule")
    lines.append(r"Method & $N$ & Generation (s) & RM scoring (s) & Total (s) & Total/direct \\")
    lines.append(r"\midrule")
    for _, r in df.iterrows():
        method = "Direct generation" if r["method"] == "direct_pi0" else "BoN reranking"
        N = int(r["N_total"])
        gen = f"{r['gen_s_mean']:.2f}"
        score = "--" if r["method"] == "direct_pi0" else f"{r['score_s_mean']:.2f}"
        total = f"{r['total_s_mean']:.2f}"
        ratio = f"{r['total_over_direct']:.2f}$\\times$"
        lines.append(f"{method} & {N} & {gen} & {score} & {total} & {ratio} \\\\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)

## Load models and held-out prompts

In [ ]:
# ============================================================
# Cell 4. Load tokenizer, generator, private RM, and held-out prompts
# ============================================================

# Read master_results.csv only. No write operation is performed on this file.
master_df = pd.read_csv(MASTER_CSV)

rm_artifact = latest_ok_rm_artifact(master_df, eps=EPS_FOR_LATENCY, seed=RM_SEED_FOR_ARTIFACT)
if rm_artifact is None:
    available = master_df[master_df.get("method", "").astype(str) == "dp_rm_postproc"]
    cols = [
        c for c in [
            "method", "epsilon_target", "seed", "train_status", "eval_status", "run_id", "artifact_dir"
        ]
        if c in available.columns
    ]
    print("Available dp_rm_postproc rows:")
    print(available[cols].to_string(index=False))
    raise RuntimeError(
        f"No successful dp_rm_postproc found for eps={EPS_FOR_LATENCY}, seed={RM_SEED_FOR_ARTIFACT}."
    )

rm_run_id, rm_artifact_dir = rm_artifact
print("[OK] using private RM artifact")
print("  rm_run_id       =", rm_run_id)
print("  rm_artifact_dir =", rm_artifact_dir)

# Extra safety: artifact path must exist and this notebook must not write there.
if not rm_artifact_dir.exists():
    raise FileNotFoundError(f"RM artifact directory does not exist: {rm_artifact_dir}")

# Load tokenizer and models. This may read/create files in HF_CACHE, but not in artifacts/.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=str(HF_CACHE))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pi0 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    cache_dir=str(HF_CACHE),
).to(DEVICE)
pi0.eval()

rm = load_rm_head_only(rm_artifact_dir, tokenizer, dtype=torch.float16)
rm.eval()

# Load held-out prompts from the same HH-RLHF slice/split convention.
raw = load_dataset("Anthropic/hh-rlhf", split=f"train[:{N_TOTAL}]", cache_dir=str(HF_CACHE))
split = raw.train_test_split(test_size=float(TEST_FRAC), shuffle=True, seed=int(PROMPT_SOURCE_SEED))
test_raw = split["test"]

prompt_rows = []
seen = set()
for raw_idx in range(len(test_raw)):
    p = extract_prompt_prefix(test_raw[raw_idx]["chosen"])
    if not p:
        continue
    ph = sha1_12(p)
    if ph in seen:
        continue
    tok_len = len(tokenizer(p, add_special_tokens=False)["input_ids"])
    if tok_len > MAX_PROMPT_TOKENS_FOR_LATENCY:
        continue
    seen.add(ph)
    prompt_rows.append({
        "raw_index": int(raw_idx),
        "prompt_hash": ph,
        # prompt text is kept only in memory for generation; it is not saved to CSV.
        "prompt_text": p,
        "prompt_tokens": int(tok_len),
        "prompt_chars": int(len(p)),
    })

need = NUM_WARMUP_PROMPTS + NUM_TIMED_PROMPTS
if len(prompt_rows) < need:
    raise RuntimeError(
        f"Only found {len(prompt_rows)} eligible prompts, but need {need}. "
        "Increase MAX_PROMPT_TOKENS_FOR_LATENCY or reduce NUM_TIMED_PROMPTS."
    )

rng = np.random.RandomState(BENCHMARK_SEED)
sel = rng.choice(len(prompt_rows), size=need, replace=False)
selected = [prompt_rows[i] for i in sel]
warmup_prompts = selected[:NUM_WARMUP_PROMPTS]
timed_prompts = selected[NUM_WARMUP_PROMPTS:]

print(f"[OK] selected warmup={len(warmup_prompts)}, timed={len(timed_prompts)} held-out prompts")
print("prompt token count: mean=", np.mean([p["prompt_tokens"] for p in timed_prompts]),
      "max=", np.max([p["prompt_tokens"] for p in timed_prompts]))

config = dict(
    run_id=RUN_ID,
    root=str(ROOT),
    master_csv=str(MASTER_CSV),
    artifacts_root=str(ARTIFACTS_ROOT),
    hf_cache=str(HF_CACHE),
    model_id=MODEL_ID,
    rm_run_id=rm_run_id,
    rm_artifact_dir=str(rm_artifact_dir),
    eps_for_latency=EPS_FOR_LATENCY,
    rm_seed_for_artifact=RM_SEED_FOR_ARTIFACT,
    n_total=N_TOTAL,
    test_frac=TEST_FRAC,
    prompt_source_seed=PROMPT_SOURCE_SEED,
    num_warmup_prompts=NUM_WARMUP_PROMPTS,
    num_timed_prompts=NUM_TIMED_PROMPTS,
    max_prompt_tokens_for_latency=MAX_PROMPT_TOKENS_FOR_LATENCY,
    n_grid=N_GRID,
    t_low=T_LOW,
    t_high=T_HIGH,
    top_p=TOP_P,
    max_new_tokens=MAX_NEW_TOKENS,
    direct_temperature=DIRECT_TEMPERATURE,
    direct_top_p=DIRECT_TOP_P,
    rm_max_len=RM_MAX_LEN,
    rm_score_batch_size=RM_SCORE_BATCH_SIZE,
    gen_return_seq_batch=GEN_RETURN_SEQ_BATCH,
    benchmark_seed=BENCHMARK_SEED,
    run_dir=str(RUN_DIR),
)

with open(CONFIG_JSON, "w") as f:
    json.dump(config, f, indent=2)

print("[OK] saved config ->", CONFIG_JSON)

## Run the latency benchmark

In [ ]:
# ============================================================
# Cell 5. Run latency benchmark
# ============================================================
rows = []

# Warmup: excluded from reporting.
print("[WARMUP]")
for pinfo in warmup_prompts:
    prompt = pinfo["prompt_text"]
    _ = run_direct_once(pi0, tokenizer, prompt, seed=900000 + pinfo["raw_index"])
    comps, _, _, _ = run_bon_generation_once(pi0, tokenizer, prompt, N=2, seed_base=910000 + pinfo["raw_index"])
    _ = score_candidates(rm, tokenizer, prompt, comps)
sync_cuda()
print("[WARMUP DONE]")

# Timed runs.
print("[TIMED RUN]")
for local_id, pinfo in enumerate(timed_prompts):
    prompt = pinfo["prompt_text"]
    ph = pinfo["prompt_hash"]
    raw_idx = pinfo["raw_index"]
    print(f"prompt {local_id + 1}/{len(timed_prompts)} | hash={ph} | tokens={pinfo['prompt_tokens']}")

    # Direct generation from pi0: one candidate, no reward scoring.
    direct_seed = 1000000 + int(raw_idx)
    (direct_out, direct_tokens), direct_gen_s = timed_call(
        run_direct_once,
        pi0,
        tokenizer,
        prompt,
        direct_seed,
    )

    rows.append(dict(
        run_id=RUN_ID,
        timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
        prompt_local_id=int(local_id),
        raw_index=int(raw_idx),
        prompt_hash=ph,
        prompt_tokens=int(pinfo["prompt_tokens"]),
        prompt_chars=int(pinfo["prompt_chars"]),
        method="direct_pi0",
        epsilon=float(EPS_FOR_LATENCY),
        rm_seed=int(RM_SEED_FOR_ARTIFACT),
        rm_run_id=str(rm_run_id),
        N_total=1,
        N_low=0,
        N_high=0,
        T_low=np.nan,
        T_high=float(DIRECT_TEMPERATURE),
        top_p=float(DIRECT_TOP_P),
        max_new_tokens=int(MAX_NEW_TOKENS),
        gen_s=float(direct_gen_s),
        score_s=0.0,
        total_s=float(direct_gen_s),
        generated_tokens_total=int(direct_tokens),
        generated_tokens_mean=float(direct_tokens),
        best_score=np.nan,
        best_idx=np.nan,
        output_chars=len(direct_out),
    ))

    # BoN reranking. Each N is generated and timed separately.
    for N in N_GRID:
        seed_base = 2000000 + 10000 * int(raw_idx) + int(N)
        (gen_out, gen_s) = timed_call(
            run_bon_generation_once,
            pi0,
            tokenizer,
            prompt,
            int(N),
            int(seed_base),
        )
        comps, tok_counts, N_low, N_high = gen_out

        scores, score_s = timed_call(
            score_candidates,
            rm,
            tokenizer,
            prompt,
            comps,
        )

        best_idx = int(np.argmax(scores))
        best_score = float(scores[best_idx])
        best_text = comps[best_idx]

        rows.append(dict(
            run_id=RUN_ID,
            timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
            prompt_local_id=int(local_id),
            raw_index=int(raw_idx),
            prompt_hash=ph,
            prompt_tokens=int(pinfo["prompt_tokens"]),
            prompt_chars=int(pinfo["prompt_chars"]),
            method="bon_rerank",
            epsilon=float(EPS_FOR_LATENCY),
            rm_seed=int(RM_SEED_FOR_ARTIFACT),
            rm_run_id=str(rm_run_id),
            N_total=int(N),
            N_low=int(N_low),
            N_high=int(N_high),
            T_low=float(T_LOW),
            T_high=float(T_HIGH),
            top_p=float(TOP_P),
            max_new_tokens=int(MAX_NEW_TOKENS),
            gen_s=float(gen_s),
            score_s=float(score_s),
            total_s=float(gen_s + score_s),
            generated_tokens_total=int(np.sum(tok_counts)),
            generated_tokens_mean=float(np.mean(tok_counts)) if len(tok_counts) else np.nan,
            best_score=best_score,
            best_idx=int(best_idx),
            output_chars=len(best_text),
        ))

# Save per-prompt timings only inside RUN_DIR.
df_lat = pd.DataFrame(rows)
df_lat.to_csv(PER_PROMPT_CSV, index=False, quoting=csv.QUOTE_ALL)
print("[OK] saved per-prompt timings ->", PER_PROMPT_CSV)

## Summarize and save results

In [ ]:
# ============================================================
# Cell 6. Aggregate summary + LaTeX table
# ============================================================
df_lat = pd.read_csv(PER_PROMPT_CSV)

agg = (
    df_lat
    .groupby(["method", "N_total"], as_index=False)
    .agg(
        n_prompts=("prompt_hash", "nunique"),
        prompt_tokens_mean=("prompt_tokens", "mean"),
        gen_s_mean=("gen_s", "mean"),
        gen_s_se=("gen_s", se),
        score_s_mean=("score_s", "mean"),
        score_s_se=("score_s", se),
        total_s_mean=("total_s", "mean"),
        total_s_se=("total_s", se),
        total_s_median=("total_s", "median"),
        total_s_p90=("total_s", lambda x: float(np.percentile(x, 90))),
        generated_tokens_total_mean=("generated_tokens_total", "mean"),
        generated_tokens_mean_per_candidate=("generated_tokens_mean", "mean"),
        best_score_mean=("best_score", "mean"),
    )
)

direct_mean = float(
    agg.loc[(agg["method"] == "direct_pi0") & (agg["N_total"] == 1), "total_s_mean"].iloc[0]
)
agg["total_over_direct"] = agg["total_s_mean"] / direct_mean

order = {"direct_pi0": 0, "bon_rerank": 1}
agg["order"] = agg["method"].map(order).fillna(99)
agg = agg.sort_values(["order", "N_total"]).drop(columns=["order"])

# Save only inside RUN_DIR. No global/latest file is overwritten.
agg.to_csv(SUMMARY_CSV, index=False, quoting=csv.QUOTE_ALL)

latex_table = make_latex_table(agg)
with open(LATEX_TABLE_TXT, "w") as f:
    f.write(latex_table + "\n")

print("=== LATENCY SUMMARY ===")
print(
    agg[[
        "method", "N_total", "n_prompts",
        "gen_s_mean", "score_s_mean", "total_s_mean", "total_s_p90",
        "total_over_direct", "generated_tokens_total_mean",
    ]].to_string(index=False)
)

print("\n=== LATEX TABLE ===")
print(latex_table)

print("\nSaved files:")
print("  per prompt :", PER_PROMPT_CSV)
print("  summary    :", SUMMARY_CSV)
print("  latex table:", LATEX_TABLE_TXT)
print("  config     :", CONFIG_JSON)

## Release GPU memory

In [ ]:
# ============================================================
# Cell 7. Optional GPU memory cleanup
# ============================================================
cleanup_gpu_only(rm, pi0)
print("Freed Python/GPU model objects from this Colab session only. No Drive files were deleted.")